In [4]:

from google.colab import files
uploaded = files.upload()


Saving all files.zip to all files (1).zip


# Step 2 — Extract all ZIP files automatically

In [5]:

import os
import zipfile

main_zip = "/content/all files.zip"
extract_folder = "/content/fyp_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(main_zip, "r") as z:
    z.extractall(extract_folder)

# Extract the two ZIP files inside the main ZIP
for root, dirs, filenames in os.walk(extract_folder):
    for filename in filenames:
        if filename.lower().endswith(".zip"):
            zip_path = os.path.join(root, filename)
            destination = os.path.join(
                extract_folder,
                os.path.splitext(filename)[0]
            )
            os.makedirs(destination, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(destination)

print("All ZIP files extracted successfully.")


All ZIP files extracted successfully.


# Step 3 — Load all four datasets

In [6]:

import os
import pandas as pd

base_folder = "/content/fyp_data"
csv_files = {}

for root, dirs, filenames in os.walk(base_folder):
    for filename in filenames:
        if filename.lower().endswith(".csv"):
            key = filename.lower()
            csv_files.setdefault(key, []).append(
                os.path.join(root, filename)
            )

def load_unique_csv(filename):
    matches = csv_files.get(filename.lower(), [])

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {filename}, "
            f"but found {len(matches)}. "
            f"Check the extracted files."
        )

    return pd.read_csv(matches[0])

original_engine = load_unique_csv("engine_data.csv")
clean_engine = load_unique_csv("engine_data_clean.csv")
clean_obd = load_unique_csv("carOBD_clean.csv")

# Combine the original OBD-II driving files
obd_paths = [
    path
    for name, paths in csv_files.items()
    if name.endswith(".csv")
    for path in paths
    if "/carOBD-master/obdiidata/" in path.replace("\\", "/")
]

if not obd_paths:
    raise FileNotFoundError("Original OBD-II CSV files were not found.")

original_obd = pd.concat(
    [pd.read_csv(path) for path in sorted(obd_paths)],
    ignore_index=True
)

print("Original engine:", original_engine.shape)
print("Cleaned engine:", clean_engine.shape)
print("Original OBD-II:", original_obd.shape)
print("Cleaned OBD-II:", clean_obd.shape)


Original engine: (19535, 7)
Cleaned engine: (19535, 8)
Original OBD-II: (304299, 27)
Cleaned OBD-II: (274968, 26)


# Step 4 — Compare the original and cleaned datasets

In [7]:

datasets = {
    "Original Engine": original_engine,
    "Cleaned Engine": clean_engine,
    "Original OBD-II": original_obd,
    "Cleaned OBD-II": clean_obd
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Rows and columns:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Columns:", df.columns.tolist())



--- Original Engine ---
Rows and columns: (19535, 7)
Missing values: 0
Duplicate rows: 0
Columns: ['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp', 'Engine Condition']

--- Cleaned Engine ---
Rows and columns: (19535, 8)
Missing values: 0
Duplicate rows: 0
Columns: ['engine_rpm', 'lub_oil_pressure', 'fuel_pressure', 'coolant_pressure', 'lub_oil_temp', 'coolant_temp', 'engine_condition', 'coolant_temp_extreme']

--- Original OBD-II ---
Rows and columns: (304299, 27)
Missing values: 295424
Duplicate rows: 42537
Columns: ['ENGINE_RUN_TINE ()', 'ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'THROTTLE ()', 'ENGINE_LOAD ()', 'COOLANT_TEMPERATURE ()', 'LONG_TERM_FUEL_TRIM_BANK_1 ()', 'SHORT_TERM_FUEL_TRIM_BANK_1 ()', 'INTAKE_MANIFOLD_PRESSURE ()', 'FUEL_TANK ()', 'ABSOLUTE_THROTTLE_B ()', 'PEDAL_D ()', 'PEDAL_E ()', 'COMMANDED_THROTTLE_ACTUATOR ()', 'FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'ABSOLUTE_BAROMETRIC_PRESSURE ()', 'RELATIVE_THROTTLE_POSITION 

In [8]:
print("Remaining missing values:")
print(clean_obd.isnull().sum().sort_values(ascending=False).head(10))

print("\nEngine label counts:")
print(clean_engine["engine_condition"].value_counts())

print("\nCoolant extreme counts:")
print(clean_engine["coolant_temp_extreme"].value_counts())

Remaining missing values:
warm_ups_since_codes_cleared        2111
time_since_trouble_codes_cleared    2111
vehicle_speed                          0
throttle                               0
engine_load                            0
coolant_temperature                    0
long_term_fuel_trim_bank_1             0
short_term_fuel_trim_bank_1            0
engine_run_time                        0
engine_rpm                             0
dtype: int64

Engine label counts:
engine_condition
1    12317
0     7218
Name: count, dtype: int64

Coolant extreme counts:
coolant_temp_extreme
0    19533
1        2
Name: count, dtype: int64


# emaining missing values remove-save new cleaned OBD-II

In [9]:
columns_to_drop = [
    "warm_ups_since_codes_cleared",
    "time_since_trouble_codes_cleared"
]

obd_final = clean_obd.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Final OBD-II shape:", obd_final.shape)
print("Missing values:", obd_final.isnull().sum().sum())

obd_final.to_csv(
    "/content/carOBD_final.csv",
    index=False
)

print("Saved successfully!")

Final OBD-II shape: (274968, 24)
Missing values: 0
Saved successfully!
